# Qwen3-4B A-natural seed 2024 — max_seq 1792

Mục tiêu: giữ 100% token (guard 960 làm mất 3,2% A).

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'no GPU'
p = torch.cuda.get_device_properties(0)
print('gpu', p.name, round(p.total_memory/1e9, 1), 'GB')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], capture_output=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [ ]:
DATA = None
for c in Path('/kaggle/input').rglob('units_A.parquet'):
    DATA = c.parent; break
assert DATA, 'không thấy tiki-matrix-data'
ROOT = Path('/kaggle/working/matrix_data'); ROOT.mkdir(parents=True, exist_ok=True)
for n in ('A','B'):
    (ROOT/f'processed_{n}').mkdir(parents=True, exist_ok=True)
    shutil.copy2(DATA/f'units_{n}.parquet', ROOT/f'processed_{n}/units.parquet')
    shutil.copy2(DATA/f'eval_{n}.parquet', ROOT/f'eval_{n}.parquet')
    shutil.copy2(DATA/f'meta_{n}.json', ROOT/f'meta_{n}.json')
TOOLS = None
for c in Path('/kaggle/input').rglob('train_weight_quota_gpu.py'):
    try:
        if 'eval-max-seq' in c.read_text():
            TOOLS = c.parent; break
    except Exception: continue
assert TOOLS, 'không thấy trainer bản mới'
MODEL = None
for cfg in Path('/kaggle/input').rglob('config.json'):
    try: d = json.loads(cfg.read_text())
    except Exception: continue
    if d.get('hidden_size') == 2560 and d.get('num_hidden_layers') == 36:
        MODEL = str(cfg.parent); break
assert MODEL, 'không thấy Qwen3-4B'
print('DATA', DATA, '| TOOLS', TOOLS, '| MODEL', MODEL)


In [ ]:
OUT = Path('/kaggle/working/out'); OUT.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, str(TOOLS/'train_weight_quota_gpu.py'),
       '--dataset', 'A', '--data-root', str(ROOT), '--out', str(OUT),
       '--model', MODEL, '--seed', '2024', '--tokens', '4000000', '--steps', '500',
       '--configs', 'natural', '--micro-batch', '2', '--max-seq', '1792',
       '--eval-max-seq', '1792', '--dtype', 'bf16', '--save-every', '25']
print('RUN', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)
src = OUT/'hyp_natural_seed2024.parquet'
if src.exists():
    src.rename(OUT/'hyp_qwen4b_a_natural_seed2024_1792.parquet')
print('DONE')
